# 00 - HITL Data Preparation

Splits the **cleaned** tweets dataset into four non-overlapping partitions:
- **LLM Bootstrap** (~10 000 tweets): labelled by an LLM in `01_llm_bootstrap_labelling.ipynb`
- **Base** (~100 000 tweets): reserve pool / source for any human seed labelling
- **HITL batches** (~200 000 tweets, 4 × 50 000): iterative human review
- **Final Inference** (remainder): classified by the final model in notebook 03

All partitions are mutually disjoint. A `partition_ids.pkl` manifest is written so any
downstream notebook can verify which subset a tweet belongs to.

Run this notebook **once** at the start of the project, before any of `01`, `02`, `03`.

In [ ]:
import os
from pathlib import Path
import sys

# --- ENVIRONMENT SWITCH ---
# True  → local machine with Google Drive Desktop mounted
# False → Google Colab cloud
RUNNING_LOCALLY = False

# --- DATASET TYPE ---
# 'AI'  → AItrust_twits_pruned_dict.json    → Partitioned Data/AI Data/
# 'Art' → AItrust_Art_pruned_twit_dict.json → Partitioned Data/Art Data/
DATASET_TYPE = 'AI'

if RUNNING_LOCALLY:
    _REPO_ROOT = str(Path(os.getcwd()).resolve().parents[1])
    if _REPO_ROOT not in sys.path:
        sys.path.insert(0, _REPO_ROOT)
    BASE_PATH = Path('/Volumes/GoogleDrive/My Drive/Colab Projects/AI Public Trust')
else:
    from google.colab import drive
    drive.mount('/content/drive')
    BASE_PATH = Path('/content/drive/My Drive/Colab Projects/AI Public Trust')

# Pre-compute critical paths
twits_folder          = BASE_PATH / 'Raw Data/Twits/'
test_folder           = BASE_PATH / 'Raw Data/'
datasets_folder       = BASE_PATH / 'Data Sets'
cleanedds_folder      = BASE_PATH / 'Data Sets/Cleaned Data'
partitioned_folder    = cleanedds_folder / 'Partitioned Data' / f'{DATASET_TYPE} Data'
networks_folder       = BASE_PATH / 'Data Sets/Networks/'
literature_folder     = BASE_PATH / 'Literature/'
topic_models_folder   = BASE_PATH / 'Models/Topic Modeling/'
classifiers_folder    = BASE_PATH / 'Models/Classifiers/'
classifiers_folder.mkdir(parents=True, exist_ok=True)
hitl_folder = datasets_folder / 'Classifiers_Data' / 'HITL'

In [ ]:
import os
if not RUNNING_LOCALLY:
    print('Running Colab setup...')
else:
    print('Running locally: skipping Colab setup.')

In [ ]:
import pickle
import numpy as np
import pandas as pd
from pathlib import Path

## Load the Pruned Tweet Dict

Source: `cleanedds_folder / 'AItrust_twits_pruned_dict.json'` — the JSONL output of
`notebooks/02_Processing/02_sanity_check_and_network_generation.ipynb`. Each line is one
tweet with the standard Twitter API v2 schema: `id`, `text`, `processed_text`,
`created_at`, `type`, `public_metrics` (nested), `referenced_tweets`, ...

Toggle `USE_TEST_DATA = True` to load the smaller `*_test.json` variant for development.

In [ ]:
USE_TEST_DATA = False  # True → AItrust_twits_pruned_dict_test.json (development); False → full dataset

_PRUNED_DICT_NAMES = {
    'AI':  ('AItrust_twits_pruned_dict_test.json'    if USE_TEST_DATA else 'AItrust_twits_pruned_dict.json'),
    'Art': ('AItrust_Art_pruned_twit_dict_test.json' if USE_TEST_DATA else 'AItrust_Art_pruned_twit_dict.json'),
}
assert DATASET_TYPE in _PRUNED_DICT_NAMES, f'Unknown DATASET_TYPE {DATASET_TYPE!r}. Choose "AI" or "Art".'
PRUNED_DICT_NAME = _PRUNED_DICT_NAMES[DATASET_TYPE]
PRUNED_DICT_PATH = cleanedds_folder / PRUNED_DICT_NAME

assert PRUNED_DICT_PATH.exists(), (
    f'Pruned dict not found: {PRUNED_DICT_PATH}. '
    f'Run notebooks/02_Processing/02_sanity_check_and_network_generation.ipynb first.'
)

print(f'Loading {PRUNED_DICT_PATH}...')
df = pd.read_json(PRUNED_DICT_PATH, lines=True)
print(f'Loaded {len(df):,} tweets')
print(f'Columns: {list(df.columns)}')

## Normalise Columns

Pull `likes` and `retweets` out of the nested `public_metrics` dict, cast `id` to string
(Twitter snowflake IDs exceed 2^53 and lose precision as Python floats), and keep the
columns the HITL pipeline needs:

- `id`, `likes`, `retweets` — identifier and engagement.
- `text` — raw tweet body, kept as-is for transformer models that were pre-trained on
  near-raw tweets (e.g. `cardiffnlp/twitter-roberta-base`).
- `processed_text` — lowercase / URL-stripped / @-mention-stripped / RT-marker-stripped
  text written by `02_sanity_check_and_network_generation.ipynb`. Carried alongside
  `text` so downstream training can pick whichever input the chosen model handles
  best (classical BoW / sentence-embedding pipelines typically do better on
  `processed_text`).
- `type` — Twitter API v2 reference type: `original`, `retweeted`, `replied_to`, or
  `quoted`. Retweets carry duplicated text, so preserving the field lets downstream
  pipelines filter them out, weight them differently, or use it as a feature.

In [ ]:
def _pm_field(pm, key, default=0):
    return pm.get(key, default) if isinstance(pm, dict) else default

required = {'id', 'text', 'processed_text', 'type', 'public_metrics'}
missing  = required - set(df.columns)
assert not missing, (
    f'Pruned tweet dict missing expected columns: {missing}. '
    f'Re-run notebooks/02_Processing/02_sanity_check_and_network_generation.ipynb '
    f'to regenerate with the current schema.'
)

df['likes']    = df['public_metrics'].apply(lambda pm: _pm_field(pm, 'like_count', 0)).astype(int)
df['retweets'] = df['public_metrics'].apply(lambda pm: _pm_field(pm, 'retweet_count', 0)).astype(int)

df['id']             = df['id'].astype(str)
df['text']           = df['text'].astype(str)
df['processed_text'] = df['processed_text'].astype(str)
df['type']           = df['type'].astype(str)

df = df[['id', 'text', 'processed_text', 'type', 'likes', 'retweets']].copy()
df['predicted_label'] = np.nan
df['human_label']     = np.nan

print(f'Normalised: {len(df):,} tweets')
print('Type breakdown:')
print(df['type'].value_counts().to_string())

## Split Retweets Out

A retweet's `text` is bit-for-bit identical to its referenced original's. If we left
retweets in the partitionable corpus, two things would go wrong:

1. **Train/test text leakage.** The same string could land in both the training
   partitions (LLM Bootstrap, Base, HITL) and the held-out Inference partition,
   inflating measured generalisation.
2. **Wasted inference compute.** The classifier would burn GPU time predicting
   labels we already know — a retweet's correct label is by construction its
   referenced original's label.

We split `type == 'retweeted'` rows into a separate `retweets_dataset.pkl` here.
They bypass the model entirely; at merge time in `03_final_inference.ipynb`, each
retweet inherits its original's predicted (or human-confirmed) label via
`referenced_tweets_dictionary`.

`replied_to` and `quoted` carry the user's own commentary text on top of a
reference, so they remain in the partitionable corpus alongside originals.

In [ ]:
RETWEET_TYPE = 'retweeted'

retweets_df = df[df['type'] == RETWEET_TYPE].copy().reset_index(drop=True)
df          = df[df['type'] != RETWEET_TYPE].copy().reset_index(drop=True)

print(f'Retweets split out: {len(retweets_df):>9,}  → retweets_dataset.pkl (label by lookup at merge time)')
print(f'Partitionable corpus: {len(df):>9,}  → originals + replies + quotes (used for train/HITL/inference)')

## Attention-Weighted Shuffle

Tweet engagement (likes + retweets) follows a heavy-tailed (near power-law) distribution:
a small number of viral tweets carry most of the attention, while the long tail gets almost
none. A **uniform** random sample of 10 000 tweets from this corpus would consist almost
entirely of low-engagement tweets — the LLM and the classifier would never see the
discourse-shaping content.

We instead do an **attention-weighted permutation**: each tweet's selection probability is
proportional to `(likes + retweets + 1) ** SAMPLING_ALPHA`. The downstream slice still cuts
the dataframe into contiguous partitions, but the partitions are now *stratified by
influence* — the LLM Bootstrap slice (first 10 000) tends to pick up the influential head,
the base / HITL / inference slices follow with progressively lighter engagement.

`SAMPLING_ALPHA` is the smoothing exponent:
- `0.0` — uniform random (recovers the previous behaviour).
- `0.5` — square-root rule (default): head is well-represented, body still well-covered.
- `1.0` — proportional to attention: heavily concentrates on the head.

In [ ]:
SAMPLING_ALPHA = 0.5  # 0 → uniform; 0.5 → sqrt smoothing (default); 1 → proportional to attention

if SAMPLING_ALPHA > 0:
    attention = (df['likes'] + df['retweets'] + 1).astype(float)
    weights   = attention ** SAMPLING_ALPHA
    df = df.sample(frac=1, weights=weights, random_state=42).reset_index(drop=True)
    print(f'Attention-weighted shuffle (alpha={SAMPLING_ALPHA})')
else:
    df = df.sample(frac=1, random_state=42).reset_index(drop=True)
    print('Uniform shuffle (alpha=0)')

## Partition the Dataset

Slices the shuffled dataframe into four contiguous, disjoint partitions:
`LLM_BOOTSTRAP_SIZE` first, then `BASE_SIZE`, then `HITL_SIZE`, then the remainder.

In [ ]:
def _summarise(name, part):
    if not len(part):
        print(f'  {name:>14}: (empty)')
        return
    att = (part['likes'] + part['retweets']).astype(int)
    print(f'  {name:>14}: n={len(part):>9,}  median={att.median():>6.0f}  mean={att.mean():>9.1f}  '
          f'p95={att.quantile(0.95):>7.0f}  max={att.max():>9,}')

print('Engagement (likes + retweets) by partition:')
_summarise('LLM bootstrap', llm_bootstrap_df)
_summarise('Base',          base_df)
_summarise('HITL',          hitl_df)
_summarise('Inference',     inference_df)
print('Retweets (label by lookup, not partitioned for training):')
_summarise('Retweets',     retweets_df)

## Engagement Distribution by Partition

Sanity check that the attention-weighted shuffle stratified the partitions as intended.
On a heavy-tailed corpus you should see the LLM Bootstrap slice carry a much higher
median / mean / max engagement than the Inference slice.

In [ ]:
partitioned_folder.mkdir(parents=True, exist_ok=True)
hitl_folder.mkdir(parents=True, exist_ok=True)

llm_bootstrap_df.to_pickle(partitioned_folder / 'llm_bootstrap_dataset.pkl')
base_df.to_pickle(partitioned_folder / 'base_dataset.pkl')
inference_df.to_pickle(partitioned_folder / 'inference_dataset.pkl')
retweets_df.to_pickle(partitioned_folder / 'retweets_dataset.pkl')
print(f'Saved retweets_dataset.pkl ({len(retweets_df):,} tweets)')

BATCH_SIZE   = 50_000
n_batches    = int(np.ceil(len(hitl_df) / BATCH_SIZE)) if len(hitl_df) else 0
hitl_batches = {}
for i in range(n_batches):
    chunk = hitl_df.iloc[i * BATCH_SIZE:(i + 1) * BATCH_SIZE]
    out   = partitioned_folder / f'hitl_pending_batch_{i+1:02d}.pkl'
    chunk.to_pickle(out)
    hitl_batches[f'hitl_batch_{i+1:02d}'] = chunk
    print(f'Saved {out.name} ({len(chunk):,} tweets)')

## Save Partitions

In [ ]:
def _ids(d):
    return d['id'].astype(str).tolist() if 'id' in d.columns else d.index.astype(str).tolist()

partition_ids = {
    'llm_bootstrap': _ids(llm_bootstrap_df),
    'base':          _ids(base_df),
    'inference':     _ids(inference_df),
    'retweets':      _ids(retweets_df),
}
for name, batch_df in hitl_batches.items():
    partition_ids[name] = _ids(batch_df)

seen = set()
for name, ids in partition_ids.items():
    s = set(ids)
    overlap = s & seen
    assert not overlap, f'Partition {name!r} overlaps existing partitions on {len(overlap)} ids'
    seen |= s
    print(f'  {name:>16}: {len(ids):>10,} ids')
print(f'  {"TOTAL":>16}: {len(seen):>10,} unique ids — all partitions disjoint')

manifest_path = partitioned_folder / 'partition_ids.pkl'
with open(manifest_path, 'wb') as f:
    pickle.dump(partition_ids, f)
print(f'Wrote → {manifest_path}')

## Partition Manifest

Write a `partition_ids.pkl` mapping every partition name to its tweet IDs and assert
that the partitions are pairwise disjoint. Downstream notebooks can load this manifest
to verify which subset any tweet belongs to.

In [ ]:
def _ids(d):
    return d['id'].astype(str).tolist() if 'id' in d.columns else d.index.astype(str).tolist()

partition_ids = {
    'llm_bootstrap': _ids(llm_bootstrap_df),
    'base':          _ids(base_df),
    'inference':     _ids(inference_df),
}
for name, batch_df in hitl_batches.items():
    partition_ids[name] = _ids(batch_df)

seen = set()
for name, ids in partition_ids.items():
    s = set(ids)
    overlap = s & seen
    assert not overlap, f'Partition {name!r} overlaps existing partitions on {len(overlap)} ids'
    seen |= s
    print(f'  {name:>16}: {len(ids):>10,} ids')
print(f'  {"TOTAL":>16}: {len(seen):>10,} unique ids — all partitions disjoint')

manifest_path = partitioned_folder / 'partition_ids.pkl'
with open(manifest_path, 'wb') as f:
    pickle.dump(partition_ids, f)
print(f'Wrote → {manifest_path}')

## (Optional) Export Human Seed Review Batch

If you intend to run the **human-only** seed path (label 10 000 tweets by hand), this
cell exports the seed CSV. Skip it if you are using the **LLM bootstrap** path
(`01_llm_bootstrap_labelling.ipynb`) — both paths produce the same downstream artifact:
a labelled CSV that `02_hitl_training_loop.ipynb` ingests as initial training data.

In [ ]:
EXPORT_HUMAN_SEED = False  # set True to export a 10k human-seed CSV from the Base partition

if EXPORT_HUMAN_SEED:
    seed = base_df.sample(n=min(10_000, len(base_df)), random_state=42).copy()
    for col in ('text', 'processed_text'):
        if col in seed.columns:
            seed[col] = seed[col].astype(str).str.replace('\n', ' ', regex=False)

    out_path = hitl_folder / 'hitl_review_batch_00.csv'
    seed.to_csv(out_path, index=False)
    print(f'Saved → {out_path}')
else:
    print('EXPORT_HUMAN_SEED=False — skipped. Set True to export the human seed batch.')

## Disconnect from Runtime

In [ ]:
if not RUNNING_LOCALLY:
    from google.colab import runtime
    runtime.unassign()